In [1]:
import pandas as pd
import sys
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from lazypredict.Supervised import LazyClassifier
from imblearn.over_sampling import SMOTE

from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from utils.helpers import Helpers

In [2]:
helper = Helpers()
properties = helper.load_properties()

try:
    feature_engineered_file_name = properties['LOCAL']['feature_engineered_file_name']
    RANDOM_STATE = properties['models']['random_state']
except KeyError as ke:
    raise KeyError(f"Missing key in properties file: {str(ke)}") from ke

In [3]:
# Apply global settings
helper.set_global_settings()

In [4]:
file_path = helper.root_dir / "datasets" / feature_engineered_file_name

df = pd.read_csv(file_path)
df.head()

,person_age,is_female,person_education,person_income,person_home_ownership,loan_amount,loan_intent,loan_interest_rate,credit_score,previous_loan_defaults,loan_status
0,-1.41,1,3,0.13,1,1.87,2,1.62,-1.38,0,1
1,-1.89,1,0,-2.82,3,-2.33,3,0.11,-2.14,1,0
2,-0.28,1,0,-2.80,2,-0.54,1,0.67,-0.04,0,1
3,-0.98,1,2,0.35,1,1.87,1,1.39,0.89,0,1
4,-0.61,0,3,-0.04,1,1.87,1,1.10,-0.98,0,1


In [5]:
df.shape

(42531, 11)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42531 entries, 0 to 42530
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   person_age              42531 non-null  float64
 1   is_female               42531 non-null  int64  
 2   person_education        42531 non-null  int64  
 3   person_income           42531 non-null  float64
 4   person_home_ownership   42531 non-null  int64  
 5   loan_amount             42531 non-null  float64
 6   loan_intent             42531 non-null  int64  
 7   loan_interest_rate      42531 non-null  float64
 8   credit_score            42531 non-null  float64
 9   previous_loan_defaults  42531 non-null  int64  
 10  loan_status             42531 non-null  int64  
dtypes: float64(5), int64(6)
memory usage: 3.6 MB


In [7]:
target_variable = 'loan_status'

df[target_variable].value_counts(normalize=True)

loan_status
0   0.78
1   0.22
Name: proportion, dtype: float64

In [8]:
X = df.drop(columns=[target_variable])
y = df[target_variable]

# Split the data into training (70%), validation (20%), and test (10%) sets
X_train, X_rem, y_train, y_rem = train_test_split(
    X, 
    y, 
    train_size=0.7, 
    stratify=y,
    random_state=RANDOM_STATE
)

X_val, X_test, y_val, y_test = train_test_split(
    X_rem, 
    y_rem, 
    train_size=2/3, 
    stratify=y_rem,
    random_state=RANDOM_STATE
)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Validation set size: {X_val.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")

Training set size: 29771 samples
Validation set size: 8506 samples
Test set size: 4254 samples


In [9]:
# Save validation and test sets to CSV files
val_set = pd.concat([X_val, y_val.reset_index(drop=True)], axis=1)
test_set = pd.concat([X_test, y_test.reset_index(drop=True)], axis=1)

parent_dir = helper.root_dir / "datasets"
parent_dir.mkdir(parents=True, exist_ok=True)

val_set.to_csv(parent_dir / "validation_set.csv", index=False)
test_set.to_csv(parent_dir / "test_set.csv", index=False)

In [10]:
# Calculate distribution before SMOTE
freq_y_train = y_train.value_counts(normalize=True) * 100

# Apply SMOTE to training set
smote = SMOTE(random_state=RANDOM_STATE)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

# Calculate distribution after SMOTE
freq_y_train_sm = pd.Series(y_train_sm).value_counts(normalize=True) * 100

# Combine into one dataframe
dist = pd.concat(
    [freq_y_train, freq_y_train_sm], 
    axis=1,
    keys=['Before SMOTE (%)', 'After SMOTE (%)']
).fillna(0).round(2)

# Print markdown table
print(f'Class distribution for {target_variable} (Train set)\n')
print(dist.to_markdown(), '\n')

Class distribution for loan_status (Train set)

|   loan_status |   Before SMOTE (%) |   After SMOTE (%) |
|--------------:|-------------------:|------------------:|
|             0 |              77.74 |                50 |
|             1 |              22.26 |                50 | 



In [ ]:
# Combine SMOTE resampled features + target
sampled_df = pd.DataFrame(X_train_sm, columns=X_train.columns)
sampled_df[target_variable] = y_train_sm

# Save to CSV
file_path = parent_dir / "sampled_train_data.csv"
sampled_df.to_csv(file_path, index=False)

In [12]:
clf = LazyClassifier(
    predictions=True, 
    random_state=RANDOM_STATE
    )

models, predictions = clf.fit(X_train_sm, X_val, y_train_sm, y_val)
models.sort_values(by=['F1 Score', 'Balanced Accuracy', 'ROC AUC', 'Time Taken'], ascending=False)

  0%|          | 0/32 [00:00<?, ?it/s]

[LightGBM] [Info] Number of positive: 23143, number of negative: 23143
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002179 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1299
[LightGBM] [Info] Number of data points in the train set: 46286, number of used features: 10
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Time Taken
Model,,,,,
XGBClassifier,0.92,0.89,0.89,0.92,0.53
LGBMClassifier,0.92,0.89,0.89,0.92,5.00
RandomForestClassifier,0.91,0.89,0.89,0.91,7.06
BaggingClassifier,0.91,0.88,0.88,0.91,1.51
ExtraTreesClassifier,0.91,0.89,0.89,0.91,2.81
DecisionTreeClassifier,0.89,0.86,0.86,0.89,0.29
SVC,0.87,0.88,0.88,0.88,32.33
KNeighborsClassifier,0.86,0.86,0.86,0.87,1.56
SGDClassifier,0.86,0.87,0.87,0.87,0.11


Top 5 models:

1. `XGBClassifier`

2. `LGBMClassifier`

3. `RandomForestClassifier`

4. `BaggingClassifier`

5. `ExtraTreesClassifier`

In [13]:
models_of_interest = [
    'XGBClassifier', 'LGBMClassifier', 'RandomForestClassifier', 'BaggingClassifier', 'ExtraTreesClassifier'
]

for model in models_of_interest:
    print('\t\t',model,'\n')
    print(classification_report(y_val, predictions[model]),'\n')

		 XGBClassifier 

              precision    recall  f1-score   support

           0       0.95      0.94      0.95      6612
           1       0.81      0.84      0.83      1894

    accuracy                           0.92      8506
   macro avg       0.88      0.89      0.89      8506
weighted avg       0.92      0.92      0.92      8506
 

		 LGBMClassifier 

              precision    recall  f1-score   support

           0       0.95      0.94      0.95      6612
           1       0.79      0.84      0.82      1894

    accuracy                           0.92      8506
   macro avg       0.87      0.89      0.88      8506
weighted avg       0.92      0.92      0.92      8506
 

		 RandomForestClassifier 

              precision    recall  f1-score   support

           0       0.96      0.93      0.94      6612
           1       0.77      0.85      0.81      1894

    accuracy                           0.91      8506
   macro avg       0.87      0.89      0.88      8506
wei